In [7]:
import pandas as pd

In [8]:
df = pd.read_csv('train.csv')

In [9]:
import numpy as np
from sklearn.preprocessing import  OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error

X = df.drop(columns=['id', 'accident_risk'])
y = df['accident_risk']

categorical_features = ['road_type', 'lighting', 'weather', 'time_of_day']
numerical_features = ['num_lanes', 'curvature', 'speed_limit', 'num_reported_accidents']
bool_features = ['road_signs_present', 'public_road', 'holiday', 'school_season']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numerical_features + bool_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

In [10]:
X_test = df.drop(columns=['id', 'accident_risk'])
y_test = df['accident_risk']

In [11]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error

rf_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(
        n_estimators=100,   
        max_depth=10,       
        random_state=42     
    ))
])

rf_model.fit(X, y)

rf_preds_optuned = rf_model.predict(X_test)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_preds_optuned))

print(f"Random Forest RMSE: {rf_rmse:.4f}")

Random Forest RMSE: 0.0556


In [12]:
final_df = pd.read_csv('test.csv')

final_preds_optuned = rf_model.predict(final_df.drop(columns=['id']))

submission = pd.DataFrame({
    'id': final_df['id'],
    'accident_risk': final_preds_optuned
})

submission.to_csv('submission_forest.csv', index=False)

# Random_Forest RMSE 0.0559